In [1]:
import os

BASE_PATH = "/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets"  
# adjust if your exact folder name differs — the one you gave was:
# /kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets

# ---- Step 2: List files ----
for root, dirs, files in os.walk(BASE_PATH):
    for f in files:
        print(os.path.join(root, f))

# ---- Step 3: CoNLL parser ----
def parse_conll(file_path):
    sentences, tags = [], []
    tokens, tag_seq = [], []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "":
                if tokens:
                    sentences.append(tokens)
                    tags.append(tag_seq)
                    tokens, tag_seq = [], []
            else:
                parts = line.split("\t")
                if len(parts) < 2:
                    continue
                token, tag = parts[0], parts[-1]
                tokens.append(token)
                tag_seq.append(tag)
    if tokens:
        sentences.append(tokens)
        tags.append(tag_seq)
    return sentences, tags

hindi_train_path = os.path.join(BASE_PATH, "Hindi", "Hindi-train.txt")
hindi_dev_path   = os.path.join(BASE_PATH, "Hindi", "Hindi-dev.txt")
hindi_test_path  = os.path.join(BASE_PATH, "Hindi", "Hindi-test.txt")

train_sents, train_tags = parse_conll(hindi_train_path)
dev_sents, dev_tags     = parse_conll(hindi_dev_path)
test_sents, test_tags   = parse_conll(hindi_test_path)

print(f"Train sentences: {len(train_sents)}  (expected 11076)")
print(f"Dev sentences:   {len(dev_sents)}  (expected 1389)")
print(f"Test sentences:  {len(test_sents)}  (expected 1389)")

# ---- Step 4: Label distribution + error check ----
from collections import Counter

all_tags = [t for seq in train_tags for t in seq]
tag_counts = Counter(all_tags)
print("\nLabel distribution (train):")
for tag, count in sorted(tag_counts.items(), key=lambda x: -x[1]):
    print(f"  {tag}: {count}")

# Check for known error patterns
valid_prefixes = ("B-", "I-", "O")
valid_types = {"NEP", "NEL", "NEO", "NEAR", "NEN", "NETI"}

bad_tags = set()
for tag in tag_counts:
    if tag == "O":
        continue
    if not tag.startswith(("B-", "I-")):
        bad_tags.add(tag)
    else:
        ttype = tag.split("-", 1)[1]
        if ttype not in valid_types:
            bad_tags.add(tag)

print(f"\nBad/unexpected tags found: {bad_tags if bad_tags else 'None — data is clean'}")

# Sample check
print("\nFirst training example:")
for tok, tag in zip(train_sents[0], train_tags[0]):
    print(f"  {tok}\t{tag}")

/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Hindi/Hindi-dev.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Hindi/Hindi-train.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Hindi/Hindi-test.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Odia/Odia-test.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Odia/Odia-dev.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Odia/Odia-train.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Telugu/Telugu-train.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Telugu/Telugu-dev.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-for-indian-languages/Datasets/Telugu/Telugu-test.txt
/kaggle/input/datasets/charumittalma25m008/ner-models-f

In [2]:
!pip install -q --use-pep517 pytorch-cr!pip install -q --use-pep517 pytorch-crf seqeval transformers

ERROR: Invalid requirement: 'pytorch-cr!pip': Expected end or semicolon (after name and no valid version specifier)
    pytorch-cr!pip
              ^


In [3]:
!pip install -q --use-pep517 pytorch-crf seqeval transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True Tesla T4


In [5]:
import os
import json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

def build_label_vocab(tag_lists):
    unique_tags = sorted(set(t for seq in tag_lists for t in seq))
    tag2id = {tag: i for i, tag in enumerate(unique_tags)}
    id2tag = {i: tag for tag, i in tag2id.items()}
    return tag2id, id2tag

tag2id, id2tag = build_label_vocab(train_tags)
print(f"Num labels: {len(tag2id)}")
print(tag2id)

with open("tag2id.json", "w", encoding="utf-8") as f:
    json.dump(tag2id, f, ensure_ascii=False, indent=2)

@torch.no_grad()
def extract_embeddings_for_split(sentences, tag_lists, tokenizer, model,
                                  max_length=256, layer="last"):
    model.eval()
    all_embeddings = []
    all_labels = []
    skipped = 0

    for words, tags in zip(sentences, tag_lists):
        encoding = tokenizer(
            words,
            is_split_into_words=True,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
        )
        word_ids = encoding.word_ids(batch_index=0)

        input_ids = encoding["input_ids"].to(DEVICE)
        attention_mask = encoding["attention_mask"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                         output_hidden_states=True)

        if layer == "last":
            hidden = outputs.last_hidden_state[0]
        elif layer == "avg_last4":
            hidden = torch.stack(outputs.hidden_states[-4:]).mean(0)[0]
        else:
            raise ValueError("layer must be 'last' or 'avg_last4'")

        hidden = hidden.cpu().numpy()

        word_vecs = {}
        for subword_idx, w_id in enumerate(word_ids):
            if w_id is None:
                continue
            word_vecs.setdefault(w_id, []).append(hidden[subword_idx])

        num_words_covered = len(word_vecs)
        if num_words_covered == 0:
            skipped += 1
            continue

        sent_embeddings = []
        sent_labels = []
        for w_id in range(num_words_covered):
            vecs = word_vecs[w_id]
            pooled = np.mean(vecs, axis=0)
            sent_embeddings.append(pooled)
            sent_labels.append(tags[w_id])

        all_embeddings.append(np.array(sent_embeddings, dtype=np.float32))
        all_labels.append(sent_labels)

    if skipped:
        print(f"  Warning: skipped {skipped} empty sentences")

    return all_embeddings, all_labels


def run_extraction(model_name, save_prefix, layer="last"):
    print(f"\n{'='*60}\nExtracting embeddings using: {model_name}\n{'='*60}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE)

    for split_name, sents, tags in [
        ("train", train_sents, train_tags),
        ("dev",   dev_sents,   dev_tags),
        ("test",  test_sents,  test_tags),
    ]:
        embs, labels = extract_embeddings_for_split(sents, tags, tokenizer, model, layer=layer)

        mismatches = sum(1 for e, l in zip(embs, labels) if e.shape[0] != len(l))
        assert mismatches == 0, f"{mismatches} alignment mismatches in {split_name}"

        out_path = f"{save_prefix}_{split_name}.npz"
        np.savez_compressed(
            out_path,
            embeddings=np.array(embs, dtype=object),
            labels=np.array(labels, dtype=object),
        )
        total_tokens = sum(len(l) for l in labels)
        print(f"  {split_name}: {len(embs)} sentences, {total_tokens} tokens -> saved to {out_path}")

    del model, tokenizer
    torch.cuda.empty_cache()


run_extraction("google/muril-base-cased", "muril_emb", layer="last")
run_extraction("xlm-roberta-base",        "xlmr_emb",  layer="last")

print("\nDone. Files created:")
for f in os.listdir("."):
    if f.endswith(".npz") or f == "tag2id.json":
        print(" ", f)

Using device: cuda
Num labels: 13
{'B-NEAR': 0, 'B-NEL': 1, 'B-NEN': 2, 'B-NEO': 3, 'B-NEP': 4, 'B-NETI': 5, 'I-NEAR': 6, 'I-NEL': 7, 'I-NEN': 8, 'I-NEO': 9, 'I-NEP': 10, 'I-NETI': 11, 'O': 12}

Extracting embeddings using: google/muril-base-cased


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  train: 11076 sentences, 262856 tokens -> saved to xlmr_emb_train.npz
  dev: 1389 sentences, 32676 tokens -> saved to xlmr_emb_dev.npz
  test: 1388 sentences, 34404 tokens -> saved to xlmr_emb_test.npz

Done. Files created:
  muril_emb_test.npz
  muril_emb_dev.npz
  muril_emb_train.npz
  tag2id.json
  xlmr_emb_dev.npz
  xlmr_emb_test.npz
  xlmr_emb_train.npz


In [6]:
"""
Tier 1 - Step B: Train BiLSTM-CRF on frozen embeddings (per backbone).

Run separately for MuRIL and XLM-R by changing EMB_PREFIX below,
or loop over both at the bottom.

Install: pip install -q pytorch-crf seqeval
"""

import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF
from seqeval.metrics import classification_report, f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open("tag2id.json", "r", encoding="utf-8") as f:
    tag2id = json.load(f)
id2tag = {v: k for k, v in tag2id.items()}
NUM_TAGS = len(tag2id)

# ---------------------------------------------------------------------
# Dataset: loads precomputed embeddings + labels from .npz
# ---------------------------------------------------------------------
class EmbeddingNERDataset(Dataset):
    def __init__(self, npz_path, tag2id):
        data = np.load(npz_path, allow_pickle=True)
        self.embeddings = data["embeddings"]  # array of (seq_len, hidden_dim) arrays
        self.labels = data["labels"]          # array of list[str]
        self.tag2id = tag2id

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        emb = torch.tensor(self.embeddings[idx], dtype=torch.float32)
        label_ids = torch.tensor(
            [self.tag2id[t] for t in self.labels[idx]], dtype=torch.long
        )
        return emb, label_ids


def collate_fn(batch):
    embs, labels = zip(*batch)
    lengths = [e.shape[0] for e in embs]
    max_len = max(lengths)
    hidden_dim = embs[0].shape[1]

    padded_embs = torch.zeros(len(embs), max_len, hidden_dim)
    padded_labels = torch.zeros(len(embs), max_len, dtype=torch.long)
    mask = torch.zeros(len(embs), max_len, dtype=torch.bool)

    for i, (e, l) in enumerate(zip(embs, labels)):
        seq_len = e.shape[0]
        padded_embs[i, :seq_len] = e
        padded_labels[i, :seq_len] = l
        mask[i, :seq_len] = 1

    return padded_embs, padded_labels, mask


# ---------------------------------------------------------------------
# Model: BiLSTM -> Linear -> CRF
# ---------------------------------------------------------------------
class BiLSTM_CRF(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_tags, num_layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, embeddings, labels=None, mask=None):
        lstm_out, _ = self.lstm(embeddings)
        lstm_out = self.dropout(lstm_out)
        emissions = self.fc(lstm_out)

        if labels is not None:
            loss = -self.crf(emissions, labels, mask=mask, reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)


# ---------------------------------------------------------------------
# Train / eval loops
# ---------------------------------------------------------------------
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for embs, labels, mask in loader:
        embs, labels, mask = embs.to(DEVICE), labels.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()
        loss = model(embs, labels=labels, mask=mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, id2tag):
    model.eval()
    all_preds, all_trues = [], []
    for embs, labels, mask in loader:
        embs, mask = embs.to(DEVICE), mask.to(DEVICE)
        pred_ids = model(embs, mask=mask)  # list of lists, variable length per sample
        for i, seq_pred in enumerate(pred_ids):
            seq_len = mask[i].sum().item()
            true_ids = labels[i][:seq_len].tolist()
            all_preds.append([id2tag[p] for p in seq_pred])
            all_trues.append([id2tag[t] for t in true_ids])
    report = classification_report(all_trues, all_preds, digits=4)
    f1 = f1_score(all_trues, all_preds)
    return f1, report, all_preds, all_trues


def run_pipeline(emb_prefix, hidden_dim=256, num_layers=1, lr=1e-3,
                  batch_size=16, epochs=15, patience=3):
    print(f"\n{'='*60}\nTraining BiLSTM-CRF on: {emb_prefix}\n{'='*60}")

    train_ds = EmbeddingNERDataset(f"{emb_prefix}_train.npz", tag2id)
    dev_ds   = EmbeddingNERDataset(f"{emb_prefix}_dev.npz",   tag2id)
    test_ds  = EmbeddingNERDataset(f"{emb_prefix}_test.npz",  tag2id)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    dev_loader   = DataLoader(dev_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    input_dim = train_ds.embeddings[0].shape[1]
    model = BiLSTM_CRF(input_dim, hidden_dim, NUM_TAGS, num_layers=num_layers).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_dev_f1 = 0.0
    best_state = None
    no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        dev_f1, _, _, _ = evaluate(model, dev_loader, id2tag)
        print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | dev_f1={dev_f1:.4f}")

        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
                break

    model.load_state_dict(best_state)
    test_f1, test_report, test_preds, test_trues = evaluate(model, test_loader, id2tag)

    print(f"\nBest dev F1: {best_dev_f1:.4f}")
    print(f"Test F1: {test_f1:.4f}\n")
    print(test_report)

    torch.save(model.state_dict(), f"{emb_prefix}_bilstm_crf_best.pt")

    return {
        "backbone": emb_prefix,
        "best_dev_f1": best_dev_f1,
        "test_f1": test_f1,
        "test_report": test_report,
    }


# ---------------------------------------------------------------------
# Run for both backbones, collect results for the comparison table
# ---------------------------------------------------------------------
results = []
for prefix in ["muril_emb", "xlmr_emb"]:
    res = run_pipeline(prefix)
    results.append(res)

with open("tier1_results_summary.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\n\nFINAL TIER 1 SUMMARY")
for r in results:
    print(f"  {r['backbone']:>12} -> test F1 = {r['test_f1']:.4f}")


Training BiLSTM-CRF on: muril_emb


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch  1 | train_loss=7.1438 | dev_f1=0.5769
Epoch  2 | train_loss=2.5558 | dev_f1=0.7210
Epoch  3 | train_loss=1.8189 | dev_f1=0.7680
Epoch  4 | train_loss=1.5010 | dev_f1=0.7425
Epoch  5 | train_loss=1.3192 | dev_f1=0.7887
Epoch  6 | train_loss=1.1886 | dev_f1=0.7856
Epoch  7 | train_loss=1.1018 | dev_f1=0.7983
Epoch  8 | train_loss=1.0217 | dev_f1=0.7971
Epoch  9 | train_loss=0.9563 | dev_f1=0.7958
Epoch 10 | train_loss=0.8901 | dev_f1=0.8084
Epoch 11 | train_loss=0.8435 | dev_f1=0.8033
Epoch 12 | train_loss=0.7856 | dev_f1=0.8007
Epoch 13 | train_loss=0.7346 | dev_f1=0.8082
Early stopping at epoch 13 (no improvement for 3 epochs)

Best dev F1: 0.8084
Test F1: 0.8172

              precision    recall  f1-score   support

        NEAR     0.6154    0.2712    0.3765        59
         NEL     0.8412    0.8826    0.8614       264
         NEN     0.9289    0.8978    0.9131       597
         NEO     0.6864    0.4551    0.5473       178
         NEP     0.8901    0.9444    0.9164      

In [7]:
!pip install -q psutil

In [8]:
"""
Resource tracking utilities — add this as a cell BEFORE your training
script, then wrap run_pipeline() calls with track_run() as shown below.

Captures: wall-clock training time, peak GPU memory, trainable vs total
params, model disk size, and per-epoch timing. Works on Kaggle T4/P100.
"""

import time
import os
import json
import torch
import psutil

def get_gpu_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0.0

def reset_gpu_memory_tracking():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def get_model_disk_size_mb(state_dict_path):
    if os.path.exists(state_dict_path):
        return os.path.getsize(state_dict_path) / (1024 ** 2)
    return None

def get_cpu_ram_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)


class RunTracker:
    """Wrap a training run to capture all resource metrics automatically."""
    def __init__(self, run_name):
        self.run_name = run_name
        self.epoch_times = []
        self.metrics = {"run_name": run_name}

    def __enter__(self):
        reset_gpu_memory_tracking()
        self.start_time = time.time()
        self.start_cpu_ram = get_cpu_ram_mb()
        return self

    def log_epoch(self, epoch_start_time):
        self.epoch_times.append(time.time() - epoch_start_time)

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.metrics["total_training_time_sec"] = round(time.time() - self.start_time, 2)
        self.metrics["avg_epoch_time_sec"] = round(
            sum(self.epoch_times) / len(self.epoch_times), 2
        ) if self.epoch_times else None
        self.metrics["num_epochs_run"] = len(self.epoch_times)
        self.metrics["peak_gpu_memory_mb"] = round(get_gpu_memory_mb(), 2)
        self.metrics["peak_cpu_ram_mb"] = round(get_cpu_ram_mb(), 2)
        self.metrics["gpu_name"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"

    def log_model_stats(self, model, state_dict_path=None):
        total, trainable = count_parameters(model)
        self.metrics["total_params"] = total
        self.metrics["trainable_params"] = trainable
        self.metrics["trainable_pct"] = round(100 * trainable / total, 4) if total else None
        if state_dict_path:
            self.metrics["model_disk_size_mb"] = round(
                get_model_disk_size_mb(state_dict_path) or 0, 2
            )

    def log_extra(self, key, value):
        self.metrics[key] = value

    def save(self, path=None):
        path = path or f"{self.run_name}_resource_metrics.json"
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.metrics, f, indent=2)
        print(f"\nResource metrics saved to {path}:")
        for k, v in self.metrics.items():
            print(f"  {k}: {v}")


# ---------------------------------------------------------------------
# MODIFIED run_pipeline: same logic as before, now wrapped with tracking.
# Replace your existing run_pipeline() with this version.
# ---------------------------------------------------------------------
def run_pipeline_tracked(emb_prefix, hidden_dim=256, num_layers=1, lr=1e-3,
                          batch_size=16, epochs=15, patience=3):
    print(f"\n{'='*60}\nTraining BiLSTM-CRF on: {emb_prefix}\n{'='*60}")

    tracker = RunTracker(f"tier1_{emb_prefix}")

    with tracker:
        train_ds = EmbeddingNERDataset(f"{emb_prefix}_train.npz", tag2id)
        dev_ds   = EmbeddingNERDataset(f"{emb_prefix}_dev.npz",   tag2id)
        test_ds  = EmbeddingNERDataset(f"{emb_prefix}_test.npz",  tag2id)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
        dev_loader   = DataLoader(dev_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
        test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

        input_dim = train_ds.embeddings[0].shape[1]
        model = BiLSTM_CRF(input_dim, hidden_dim, NUM_TAGS, num_layers=num_layers).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        best_dev_f1 = 0.0
        best_state = None
        no_improve = 0

        for epoch in range(1, epochs + 1):
            epoch_start = time.time()
            train_loss = train_one_epoch(model, train_loader, optimizer)
            tracker.log_epoch(epoch_start)

            dev_f1, _, _, _ = evaluate(model, dev_loader, id2tag)
            print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | dev_f1={dev_f1:.4f} "
                  f"| epoch_time={tracker.epoch_times[-1]:.1f}s")

            if dev_f1 > best_dev_f1:
                best_dev_f1 = dev_f1
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break

        model.load_state_dict(best_state)
        model_path = f"{emb_prefix}_bilstm_crf_best.pt"
        torch.save(model.state_dict(), model_path)

        tracker.log_model_stats(model, model_path)

        # Inference latency: measure test-set predict time separately (not training)
        infer_start = time.time()
        test_f1, test_report, test_preds, test_trues = evaluate(model, test_loader, id2tag)
        infer_time = time.time() - infer_start
        tracker.log_extra("test_inference_time_sec", round(infer_time, 3))
        tracker.log_extra("test_inference_time_per_sentence_ms",
                           round(1000 * infer_time / len(test_ds), 3))
        tracker.log_extra("best_dev_f1", round(best_dev_f1, 4))
        tracker.log_extra("test_f1", round(test_f1, 4))

    tracker.save()

    print(f"\nBest dev F1: {best_dev_f1:.4f}")
    print(f"Test F1: {test_f1:.4f}\n")
    print(test_report)

    return {
        "backbone": emb_prefix,
        "best_dev_f1": best_dev_f1,
        "test_f1": test_f1,
        "test_report": test_report,
        "resource_metrics": tracker.metrics,
    }


# ---------------------------------------------------------------------
# Run both backbones with tracking, save combined summary
# ---------------------------------------------------------------------
results = []
for prefix in ["muril_emb", "xlmr_emb"]:
    res = run_pipeline_tracked(prefix)
    results.append(res)

with open("tier1_results_summary_with_resources.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\n\nFINAL TIER 1 SUMMARY (accuracy + resource cost)")
print(f"{'Backbone':>12} | {'Test F1':>8} | {'Train time(s)':>14} | {'Peak GPU(MB)':>13} | {'Trainable params':>18}")
for r in results:
    m = r["resource_metrics"]
    print(f"{r['backbone']:>12} | {r['test_f1']:>8.4f} | {m['total_training_time_sec']:>14.1f} | "
          f"{m['peak_gpu_memory_mb']:>13.1f} | {m['trainable_params']:>18,}")


Training BiLSTM-CRF on: muril_emb
Epoch  1 | train_loss=6.9616 | dev_f1=0.5970 | epoch_time=40.4s
Epoch  2 | train_loss=2.6034 | dev_f1=0.7407 | epoch_time=39.9s
Epoch  3 | train_loss=1.8371 | dev_f1=0.7500 | epoch_time=39.9s
Epoch  4 | train_loss=1.5070 | dev_f1=0.7632 | epoch_time=39.9s
Epoch  5 | train_loss=1.3063 | dev_f1=0.7915 | epoch_time=39.9s
Epoch  6 | train_loss=1.1864 | dev_f1=0.7895 | epoch_time=39.7s
Epoch  7 | train_loss=1.1025 | dev_f1=0.7873 | epoch_time=39.9s
Epoch  8 | train_loss=1.0138 | dev_f1=0.7925 | epoch_time=39.8s
Epoch  9 | train_loss=0.9470 | dev_f1=0.8029 | epoch_time=40.1s
Epoch 10 | train_loss=0.8965 | dev_f1=0.7869 | epoch_time=40.1s
Epoch 11 | train_loss=0.8297 | dev_f1=0.7902 | epoch_time=40.2s
Epoch 12 | train_loss=0.7847 | dev_f1=0.8084 | epoch_time=40.2s
Epoch 13 | train_loss=0.7337 | dev_f1=0.7846 | epoch_time=40.2s
Epoch 14 | train_loss=0.6745 | dev_f1=0.8012 | epoch_time=39.9s
Epoch 15 | train_loss=0.6281 | dev_f1=0.8017 | epoch_time=40.3s
Early

In [9]:
"""
Tier 2: Full fine-tuning of XLM-RoBERTa-base on Hindi NER.
CORRECTED VERSION - proper handling of padding labels for CRF.

Key fix:
- Replace -100 (CrossEntropyLoss ignore_index) with 0 (valid "O" tag index)
  at padding positions. The CRF uses the mask to ignore these positions,
  so the actual value doesn't matter as long as it's a valid tag index.

Install (if not already):
    !pip install -q transformers seqeval pytorch-crf psutil
"""

import os
import json
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from seqeval.metrics import classification_report, f1_score
from torchcrf import CRF

# ---------------------------------------------------------------------
# Reuse your existing tag2id mapping from Tier 1
# ---------------------------------------------------------------------
with open("tag2id.json", "r", encoding="utf-8") as f:
    tag2id = json.load(f)
id2tag = {v: k for k, v in tag2id.items()}
NUM_TAGS = len(tag2id)

# Find the index of "O" tag (or use 0 as dummy if "O" doesn't exist)
O_TAG_INDEX = tag2id.get("O", 0)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"O tag index: {O_TAG_INDEX}")

# ---------------------------------------------------------------------
# Dataset: tokenizes raw sentences on-the-fly, returns token-level labels
# ---------------------------------------------------------------------
class NERDataset(Dataset):
    def __init__(self, sentences, tags, tokenizer, max_length=256):
        self.sentences = sentences
        self.tags = tags
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        labels = self.tags[idx]

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        word_ids = encoding.word_ids(batch_index=0)
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # CRITICAL FIX: Use O_TAG_INDEX (valid index) instead of -100 for padding
        # The CRF will use the mask to ignore these positions anyway
        label_ids = torch.full_like(input_ids, O_TAG_INDEX, dtype=torch.long)
        for i, w_id in enumerate(word_ids):
            if w_id is not None:
                label_ids[i] = tag2id[labels[w_id]]

        return input_ids, attention_mask, label_ids


def collate_fn(batch):
    input_ids, attention_masks, label_ids = zip(*batch)
    input_ids = torch.stack(input_ids)
    attention_masks = torch.stack(attention_masks)
    label_ids = torch.stack(label_ids)
    crf_mask = attention_masks.bool()
    return input_ids, attention_masks, label_ids, crf_mask


# ---------------------------------------------------------------------
# Model: XLM-R + linear projection + CRF (no BiLSTM this time)
# ---------------------------------------------------------------------
class XLMR_CRF(nn.Module):
    def __init__(self, model_name, num_tags, dropout=0.3):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="right")
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden_dim = self.transformer.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, mask=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        hidden = self.dropout(hidden)
        emissions = self.fc(hidden)

        if labels is not None:
            loss = -self.crf(emissions, labels, mask=mask, reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)


# ---------------------------------------------------------------------
# Train / eval loops
# ---------------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss = 0.0
    for input_ids, attention_mask, labels, mask in loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)
        mask = mask.to(DEVICE)

        optimizer.zero_grad()
        loss = model(input_ids, attention_mask, labels=labels, mask=mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, id2tag):
    model.eval()
    all_preds, all_trues = [], []
    for input_ids, attention_mask, labels, mask in loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        mask = mask.to(DEVICE)

        pred_ids = model(input_ids, attention_mask, mask=mask)
        for i, seq_pred in enumerate(pred_ids):
            seq_len = mask[i].sum().item()
            true_ids = labels[i][:seq_len].tolist()
            all_preds.append([id2tag[p] for p in seq_pred])
            all_trues.append([id2tag[t] for t in true_ids])
    report = classification_report(all_trues, all_preds, digits=4)
    f1 = f1_score(all_trues, all_preds)
    return f1, report, all_preds, all_trues


def run_finetune(model_name="xlm-roberta-base", lr=5e-5, batch_size=4,
                  epochs=10, patience=3, max_length=256):
    print(f"\n{'='*60}\nTier 2: Full fine-tuning {model_name}\n{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="right")

    train_ds = NERDataset(train_sents, train_tags, tokenizer, max_length)
    dev_ds   = NERDataset(dev_sents,   dev_tags,   tokenizer, max_length)
    test_ds  = NERDataset(test_sents,  test_tags,  tokenizer, max_length)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    dev_loader   = DataLoader(dev_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    model = XLMR_CRF(model_name, NUM_TAGS).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )

    best_dev_f1 = 0.0
    best_state = None
    no_improve = 0
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        dev_f1, _, _, _ = evaluate(model, dev_loader, id2tag)
        epoch_time = time.time() - epoch_start
        print(f"Epoch {epoch:2d} | train_loss={train_loss:.4f} | dev_f1={dev_f1:.4f} | epoch_time={epoch_time:.1f}s")

        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    model_path = "xlmr_finetuned_best.pt"
    torch.save(model.state_dict(), model_path)

    test_f1, test_report, test_preds, test_trues = evaluate(model, test_loader, id2tag)
    total_time = time.time() - start_time

    print(f"\nBest dev F1: {best_dev_f1:.4f}")
    print(f"Test F1: {test_f1:.4f} (total training time: {total_time:.1f}s)")
    print(test_report)

    return {
        "tier": "tier2_full_finetune",
        "model": model_name,
        "best_dev_f1": round(best_dev_f1, 4),
        "test_f1": round(test_f1, 4),
        "test_report": test_report,
        "total_training_time_sec": round(total_time, 2),
        "model_path": model_path,
    }


# ---------------------------------------------------------------------
# Run it
# ---------------------------------------------------------------------
tier2_results = run_finetune()

with open("tier2_results.json", "w", encoding="utf-8") as f:
    json.dump(tier2_results, f, ensure_ascii=False, indent=2)

print("\nTier 2 results saved to tier2_results.json")

Using device: cuda
O tag index: 12

Tier 2: Full fine-tuning xlm-roberta-base


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch  1 | train_loss=9.9156 | dev_f1=0.7426 | epoch_time=1376.6s
Epoch  2 | train_loss=4.7748 | dev_f1=0.7807 | epoch_time=1374.1s
Epoch  3 | train_loss=3.8170 | dev_f1=0.7650 | epoch_time=1373.3s
Epoch  4 | train_loss=3.2167 | dev_f1=0.7252 | epoch_time=1374.8s
Epoch  5 | train_loss=2.5720 | dev_f1=0.8090 | epoch_time=1373.2s
Epoch  6 | train_loss=1.9996 | dev_f1=0.8060 | epoch_time=1376.9s
Epoch  7 | train_loss=1.6907 | dev_f1=0.7938 | epoch_time=1374.7s
Epoch  8 | train_loss=1.1950 | dev_f1=0.8253 | epoch_time=1372.8s
Epoch  9 | train_loss=0.8984 | dev_f1=0.8250 | epoch_time=1374.0s
Epoch 10 | train_loss=0.6318 | dev_f1=0.8239 | epoch_time=1372.2s

Best dev F1: 0.8253
Test F1: 0.7901 (total training time: 13773.6s)
              precision    recall  f1-score   support

        NEAR     0.4462    0.4000    0.4218       145
         NEL     0.8277    0.8611    0.8441       396
         NEN     0.9365    0.9077    0.9219       780
         NEO     0.6921    0.5742    0.6276       364
